# Objetivo 1 (Operaciones CRUD).

In [1]:
!pip install -r ../requirements.txt

In [2]:
import sys
from pathlib import Path

In [3]:
obj_1_path = Path(".").absolute().parent / "Objetivo 1"
sys.path.append(str(obj_1_path))
from crud_handler import RedisCRUDHandler

In [4]:
import redis
import pandas as pd
from tqdm import tqdm

In [5]:
path = "../data/cards_final_with_xp.csv"
df = pd.read_csv(path)

In [6]:
df.columns

Index(['code', 'name', 'text', 'type_code', 'traits', 'pack_code',
       'illustrator', 'image_url', 'xp', 'faction_code'],
      dtype='str')

In [7]:
df

,code,name,text,type_code,traits,pack_code,illustrator,image_url,xp,faction_code
0,60401,Jacqueline Fine,[reaction] When an investigator at your locati...,investigator,Clairvoyant,jac,Aleksander Karcz,https://arkhamdb.com/bundles/cards/60401.jpg,0,mystic
1,60402,Arbiter of Fates,Jacqueline Fine deck only. [reaction] When you...,asset,Talent,jac,Pavel Kolomeyets,https://arkhamdb.com/bundles/cards/60402.jpg,0,mystic
2,60403,Dark Future,Revelation - Put Dark Future into play in your...,treachery,Omen|Endtimes,jac,Matt Bradbury,https://arkhamdb.com/bundles/cards/60403.jpg,0,neutral
3,60404,Nihilism,Revelation - Put Nihilism into play in your th...,treachery,Madness,jac,Sara Biddle,https://arkhamdb.com/bundles/cards/60404.jpg,0,neutral
4,60406,Scrying Mirror,Uses (4 secrets). [reaction] After a skill tes...,asset,Item|Charm,jac,Drazenka Kimpel,https://arkhamdb.com/bundles/cards/60406.jpg,0,mystic
...,...,...,...,...,...,...,...,...,...,...
5341,08129,Call for Backup,"One at a time, in any order, if you control a....",event,Favor|Synergy,eoep,Adam Schumpert,https://arkhamdb.com/bundles/cards/08129.jpg,2,neutral
5342,08130,Arm Injury,Revelation - Put Arm Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08130.jpg,0,neutral
5343,08131,Leg Injury,Revelation - Put Leg Injury into play in your ...,treachery,Injury,eoep,Robert Laskey,https://arkhamdb.com/bundles/cards/08131.jpg,0,neutral
5344,08132,Panic,Revelation - Put Panic into play in your threa...,treachery,Madness,eoep,Felicia Cano,https://arkhamdb.com/bundles/cards/08132.jpg,0,neutral


In [8]:
import redis

r = redis.Redis(decode_responses=True)

In [9]:
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.field import TextField, NumericField, TagField, VectorField

schema = (
    # Tags
    TagField("traits", separator="|"),
    TagField("faction_code", separator="|"),

    #Texto
    TextField("text"),
    TextField("name"),
)

index = IndexDefinition(prefix=["cards:"], index_type=IndexType.HASH)

In [10]:
idx_name = "idx_recomendation"

try:
    r.ft(idx_name).create_index(definition=index, fields=schema)
except Exception as e:
    r.ft(idx_name).dropindex()
    r.ft(idx_name).create_index(definition=index, fields=schema)

primera querie

In [11]:
faccion = "mystic"

from redis.commands.search.query import Query
from redis.commands.search.aggregation import AggregateRequest

q = Query("@faction_code:{mystic} @traits:{Practiced}")
r.ft(idx_name).search(q)

Result{0 total, docs: []}

segunda querie.

In [12]:
texto = "deals damage to all anemies and a investigator"
texto = texto.replace(" ", "|")

q = Query(f"@text:({texto})").with_scores()
result = r.ft(idx_name).search(q)
result.docs[0].score
result.docs[0].text

'Uses (2 ammo). [action] Spend 1 ammo: Fight. You get +3 [combat] for this attack. Instead of its standard damage, this attack deals 1 damage for each point you succeed by (to a minimum of 1, to a maximum of 5). If you fail and would damage another investigator, this attack deals 1 damage for each point you fail by (to a minimum of 1, to a maximum of 5).'

In [17]:
from sentence_transformers import SentenceTransformer

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\iocol\Desktop\practicas\redis\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iocol\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import numpy as np

def get_embeddings(text):
    embeding = model.encode(text, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
    return embeding

In [40]:
keys = r.keys("cards:*")

for key in tqdm(keys):
    card = r.hmget(key, "name", "text")
    name = card[0]
    text = card[1]
    full_text = name + "\n" + text
    embeddings = get_embeddings(full_text).tobytes()
    r.hset(key, "embedding", embeddings)

100%|██████████| 5324/5324 [01:37<00:00, 54.66it/s]


In [44]:
r_bin = redis.Redis(decode_responses=False) 
bin_emb = r_bin.hget(keys[0], "embedding")
emb = np.frombuffer(bin_emb, dtype=np.float32)
emb.shape

(384,)

In [45]:
from redis.commands.search.field import VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType

schema = (
    #vector
    VectorField("embedding", "FLAT", {"TYPE": "FLOAT32", "DIM": 384, "DISTANCE_METRIC": "COSINE"}),
)

idx = IndexDefinition(prefix=["cards:"], index_type=IndexType.HASH)
idx_name = "idx_vector"
try:
    r.ft(idx_name).create_index(definition=idx, fields=schema)
except Exception as e:
    r.ft(idx_name).dropindex()
    r.ft(idx_name).create_index(definition=idx, fields=schema)

In [47]:
from redis.commands.search.query import Query

query = Query("*=>[KNN 5 @embedding $vec_param AS vector_score]").sort_by("vector_score").return_fields("name", "text", "vector_score").dialect(2)
r.ft(idx_name).search(query, query_params={"vec_param": emb.tobytes()})

Result{5 total, docs: [Document {'id': 'cards:08625', 'payload': None, 'vector_score': '1.19209289551e-07', 'name': 'Sprawling City (v. II)', 'text': '[action] Spend two [elder_thing] keys: Remember that the team "studied the history of the Elder Things." [action] Spend two [cultist] keys: Remember that the team "discovered a hidden power." Objective - If the team has "studied the history of the Elder Things" and "discovered a hidden power," advance.'}, Document {'id': 'cards:08624', 'payload': None, 'vector_score': '0.160981476307', 'name': 'Sprawling City (v. I)', 'text': '[action] Spend two [elder_thing] keys: Remember that the team "studied the history of the Elder Things." [action] Spend two [tablet] keys: Remember that the team "discerned the origin of the Shoggoths." Objective - If the team has "studied the history of the Elder Things" and "discerned the origin of the Shoggoths," advance.'}, Document {'id': 'cards:08626', 'payload': None, 'vector_score': '0.203563213348', 'name'